# 03 · Migration Flow Forecasting

End-to-end walkthrough of the flow forecaster — the headline ML model in this project.

**Goal:** Predict country-of-origin migration flows for the next 1–5 years.

**Approach:** Prophet + lightweight LSTM ensemble, weighted by inverse validation-set MAE.

**Data:** A synthetic flows dataset is generated automatically if no real data is present, so the notebook is runnable end-to-end. Replace with USCIS / Census ACS data for real results.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve() / 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from migration_atlas.models.forecaster import (
    ForecastConfig, fit_country, prophet_forecast, lstm_forecast, _write_synthetic_flows,
)

## 1. Generate synthetic flows (or load real)

In [ ]:
flows_path = Path('../data/processed/flows.parquet')
if not flows_path.exists():
    flows_path.parent.mkdir(parents=True, exist_ok=True)
    _write_synthetic_flows(flows_path, [
        'mexico', 'india', 'china', 'philippines', 'vietnam'
    ])

flows = pd.read_parquet(flows_path)
flows.head()

In [ ]:
# Plot the historical flows
fig, ax = plt.subplots(figsize=(10, 5))
for country, sub in flows.groupby('country'):
    ax.plot(sub['year'], sub['flow'], marker='o', markersize=3, label=country)
ax.set_title('Historical annual flows (synthetic)')
ax.set_xlabel('Year')
ax.set_ylabel('Flow')
ax.legend(loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

## 2. Fit Prophet on a single country (Mexico)

In [ ]:
mex = flows[flows['country'] == 'mexico']
series = mex.set_index(pd.to_datetime(mex['year'], format='%Y'))['flow']

p_fc = prophet_forecast(series, horizon=5)
p_fc.tail(10)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(series.index, series.values, marker='o', markersize=4, label='Historical', color='#0c0a09')
ax.plot(p_fc['year'], p_fc['yhat'], color='#c2410c', label='Prophet forecast')
ax.fill_between(p_fc['year'], p_fc['yhat_lower'], p_fc['yhat_upper'],
                color='#c2410c', alpha=0.2, label='95% interval')
ax.set_title('Mexico migration flow — Prophet forecast')
ax.set_xlabel('Year')
ax.set_ylabel('Flow')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Fit LSTM on the same series

In [ ]:
cfg = ForecastConfig(use_lstm=True, lstm_epochs=300)
lstm_fc = lstm_forecast(series, horizon=5, cfg=cfg)
lstm_fc

## 4. Run the full ensemble pipeline

This validates each forecaster on the last 5 years, weights them by inverse MAE, then forecasts the next 5 years on the full series.

In [ ]:
results = {}
for country in ['mexico', 'india', 'china', 'philippines', 'vietnam']:
    sub = flows[flows['country'] == country]
    results[country] = fit_country(country, sub, cfg)
    print(f'{country}: val MAE -> {results[country].get("val_mae")}')

In [ ]:
# Plot all five with their forecasts
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, (country, res) in zip(axes.flat, results.items()):
    sub = flows[flows['country'] == country]
    ax.plot(pd.to_datetime(sub['year'], format='%Y'), sub['flow'],
            marker='o', markersize=3, color='#0c0a09', label='Historical')
    fc = res['forecast']
    ax.plot(fc['year'], fc['yhat'], color='#c2410c', label='Forecast')
    if 'yhat_lower' in fc.columns:
        ax.fill_between(fc['year'], fc['yhat_lower'], fc['yhat_upper'],
                        color='#c2410c', alpha=0.2)
    ax.set_title(country)
    ax.tick_params(axis='x', rotation=30, labelsize=8)
axes.flat[-1].axis('off')
plt.tight_layout()
plt.show()

## 5. Honest caveats

These forecasts assume the **current policy environment continues**. Structural breaks — a new restrictive law, a war, a major economic collapse — are unforecastable from historical data alone. The 95% prediction intervals widen quickly past 3 years; treat the 5-year point estimates as central tendencies, not predictions.

On real data, the ensemble typically beats either model alone by 5–15% on holdout MAE. On these synthetic flows, the gap is smaller because the data lacks real structural breaks.